# 制作复盘：可检查的汇总与恒等式

原始文件复算命令见 README。此 notebook 仅用 Python 标准库检查经投影的 summary.json；完整审计在四个脚本中。它不运行模型、联网、部署或播放媒体。输入日志未随 PR 上传，因此无本地 RUN 时只能复核已提交聚合与口径，不能重新证明原始证据。

In [1]:
from pathlib import Path
import json
p = Path("summary.json")
if not p.exists():
    p = Path("docs/reports/20261004-retrospective/summary.json")
s = json.loads(p.read_text())
print(s["sourceId"])


resi-20261004-69ba7a66


## L2 分类闭合与缓存

以原因表和逐轮表分别聚合，二者都必须与独立既有生产账本的 L2 汇总相符。缺失用量单列。

In [2]:
v = s["versions"]
r = list(v["reasonTotals"].values())
runs = v["runsSummary"]
api = sum(x["attempts"] for x in r)
tokens = sum(x["knownInputTokens"] + x["knownOutputTokens"] for x in r)
assert api == sum(x["apiAttempts"] for x in runs) == 8166
assert tokens == sum(x["inputTokens"] + x["outputTokens"] for x in runs) == 23276962
assert sum(x["usageMissing"] for x in r) == 16
policy = sum(v["reasonTotals"][k][t] for k in ["model_policy_quote_map_was_missing", "model_policy_remove_unspoken_citations"] for t in ["knownInputTokens", "knownOutputTokens"])
assert policy == 6473094
migrations = [x for x in runs if "/plugin-revision-" in x["path"]]
assert len(migrations) == 3 and all(x["apiAttempts"] == 0 and sum(x["cacheObservations"].values()) == 948 for x in migrations)
assert v["exactPayloadDuplicateSummary"]["extraSuccessfulUsageBearingCalls"] == 0
print({"apiAttempts": api, "knownTokens": tokens, "policyRepairTokens": policy, "zeroCallMigrations": len(migrations)})


{'apiAttempts': 8166, 'knownTokens': 23276962, 'policyRepairTokens': 6473094, 'zeroCallMigrations': 3}


## 音频异常与原目标

重算 u172 的局部 lag 跳升，而不是根据最长累计 lag 推断整段译文过长。最终62秒政策通过不等于8秒目标通过。

In [3]:
for locale, expected_over8 in [("zh-Hans",167),("ko",137),("es",200)]:
    x=s["audio"]["locales"][locale]
    assert x["units"] == x["asr"]["results"] == 474
    assert x["over8Seconds"] == expected_over8 and x["over62Seconds"] == 0
    assert x["asr"]["falsePositiveRate"] is None
    print(locale, "maxLagSeconds", round(x["maxLagSeconds"],6), "over8", x["over8Seconds"])
for locale, x in s["audio"]["u172Discontinuity"].items():
    u=x["initialCompactedWindow"][2]
    assert abs(u["sourceDurationSeconds"]-2.960022) < .00001
    assert u["audioDurationSeconds"] > 61 and u["priorEndLagSeconds"] < 1
    assert abs(u["lagJumpSeconds"]-(u["endLagSeconds"]-u["priorEndLagSeconds"])) < 1e-9
    print(locale, "u172 lag jump", round(u["lagJumpSeconds"],6))


zh-Hans maxLagSeconds 30.510004 over8 167
ko maxLagSeconds 18.320103 over8 137
es maxLagSeconds 24.569985 over8 200
zh-Hans u172 lag jump 58.199966
ko u172 lag jump 57.949966


## 批准、部署与HTTP身份

历史最终HTTP资产哈希通过；旧收据身份失配另行计数，设备和现场状态保留。

In [4]:
h=s["handoffs"]
assert h["initialCoverage"] == {"candidates":6,"hashBoundApproved":6,"readyEvents":5,"spokenLaunchBound":3}
spoken=[x for x in h["initialApprovalHandoffs"] if x["variant"]=="spoken"]
assert all(abs(x["approvalToQueueStartSeconds"]-920.6059)<1e-6 for x in spoken)
assert len(h["inheritedHttpCatalogMismatch"]) == 6
assert h["receiptCoverage"]["deploymentCandidateDirectoryMismatch"] == 4
assert sum(x["matchedSnapshotFiles"] for x in h["finalHttp"]) == 30
assert sum(x["range206"] for x in h["finalHttp"]) == 6
assert all(x["deviceAcceptance"]==x["venueAcceptance"]=="not_run" for x in h["finalHttp"])
assert all(x["catalogReleaseShaMatches"] and x["httpEvidenceHashMatchesRawReceipt"] and x["assetHashesMatchBothHttpReceipts"]==x["assetCount"] for x in h["finalReleaseBindings"])
print("6 approvals; 4 release bindings; 30 file hashes; 6 Range records; device/venue not_run")


6 approvals; 4 release bindings; 30 file hashes; 6 Range records; device/venue not_run


## 操作调用的分母

外层调用、exec内部静态位置和shell子进程不同。重复命令中状态大多变化，不能推导无效成本。

In [5]:
m=s["operations"]["scopes"]["main"]
assert m["outerToolCalls"] == m["functionCalls"] + m["customToolCalls"] == sum(m["categories"].values()) == 3801
assert sum(m["commandClasses"].values()) == 2784
assert m["literalCommandCount"]-m["uniqueLiteralCommands"] == m["exactRepeatedCommandOccurrences"] == 233
q=m["progressQuerySamples"]
assert sum(x["observations"] for x in q)==59 and sum(x["distinctStates"] for x in q)==56
print("3801 outer calls; 2784 static command sites; sampled polling 56 distinct states / 59 observations")


3801 outer calls; 2784 static command sites; sampled polling 56 distinct states / 59 observations
